# PM2.5 forecasting with a Darts TFTModel

Trains a single global **Temporal Fusion Transformer** across all GISTDA
stations in `clean-data_preprocess_all_stations_daily.csv`, forecasting
`pm25` 1-7 days ahead.

This notebook is the single-file version of the `pm25-tft-model/` project.
Background reading:

- TFT architecture & Darts `TFTModel` API notes: `../REFERENCE.md`
- Full feature-selection rationale + citations: `FEATURE_SELECTION.md`
  (condensed inline below, in Section 2)

Run top to bottom. Section 3 has the editable configuration (paths,
architecture, training loop).

## 1. Setup

```bash
pip install -r requirements.txt
```

(`darts[torch]>=0.46.1,<0.47`, `pandas`, `numpy`, `scikit-learn` - see
`requirements.txt`.)

In [ ]:
from __future__ import annotations

import logging
import pickle
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from darts import TimeSeries
from darts.dataprocessing.transformers import (
    MissingValuesFiller,
    Scaler,
    StaticCovariatesTransformer,
)
from darts.models import TFTModel
from darts.utils.likelihood_models.torch import QuantileRegression

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("pm25_tft")


## 2. Feature selection (condensed - see `FEATURE_SELECTION.md` for full detail + citations)

The source table has 257 columns: raw weather, hotspot/fire detections,
calendar fields, cyclical encodings, QC flags, imputation flags, lag/
rolling/diff features, and precomputed multi-horizon targets. Most of that
is either pipeline bookkeeping (not a predictor) or redundant with
something else already selected. What's kept, and why:

**Target**: `pm25`.

**Static covariate**: `station_id` only. It already uniquely identifies a
physical location (227 stations), mirroring how the TFT paper uses a
single entity ID (store ID, meter ID) rather than also feeding the
entity's separately-known attributes. Confirmed with the user
(2026-09-12): no lat/long/province on top.

**Known-future covariates** (must be knowable ahead of time - there's no
weather-forecast feed in this dataset, so only calendar-derived signals
qualify): `year_index`, `dow_sin/cos`, `month_sin/cos`, `doy_sin/cos`.
Sine/cosine pairs are used instead of the raw integer calendar columns to
avoid a false discontinuity between adjacent-but-wrapped values (e.g.
December vs. January). **Correction to `feature_manifest.yaml`**: its
`TFT_KNOWN_TIME_FEATURES` view lists `wind_direction_avg_sin/cos` as
known-future, but wind direction is measured weather with no forecast feed
- it was moved to the observed/past-only group instead (confirmed with the
user 2026-09-12).

**Past (observed-only) covariates** - trimmed from the manifest's ~33-column
`TFT_OBSERVED_FEATURES` view down to 14, removing collinear duplicates
(raw vs. log1p, avg/min/max/range, sum/mean/max of the same underlying
quantity) and columns with a documented, cited link to PM2.5:
`temperature_avg`, `temperature_range`, `humidity_avg`, `humidity_range`,
`pressure_avg`, `pressure_range`, `wind_speed_avg`,
`wind_direction_avg_sin/cos`, `log1p_rainfall`, `rain_event`,
`hotspot_present`, `log1p_hotspot_count`, `log1p_hotspot_frp_sum`.

Two data-quality findings from checking the real CSV directly (not just the
schema) shaped this list further:

- **~39 of 227 "stations" have no PM2.5 sensor at all** (pure
  meteorological stations, e.g. *"Narathiwat Weather Observing Station"*) -
  their `pm25` is 90-100% `NaN` for the whole segment. These are excluded
  as training *targets* below (`MAX_TARGET_NAN_FRAC`), not as feature
  columns.
- **`sunshine_duration` was dropped**: despite looking manageable in
  aggregate (~12% missing), it is **100% missing for 80 of 537** otherwise-
  usable station segments - no sensor installed, not scattered gaps.
  Interpolation can't recover a fully-missing column, and imputing a
  fabricated value for 15% of the data would misrepresent real conditions.
- All `*_lag_*`/`*_roll_*`/`*_diff_*` engineered columns are deliberately
  excluded, matching the manifest's own `TFT_OBSERVED_FEATURES` view: the
  TFT's LSTM encoder + attention block is designed to learn that temporal
  structure directly from the raw `input_chunk_length` window (Lim et al.
  2019, §4.3-4.4), so hand-feeding pre-computed lags would just duplicate
  it and blur variable-importance interpretability.

**Forecast horizon**: one multi-step model, `output_chunk_length=7`
(days 1-7 in a single forward pass), confirmed with the user (2026-09-12).
This is also why the precomputed `target_pm25_t1/t2/t3/t7` columns are
unused - Darts' `TFTModel` slices every horizon step out of the `pm25`
series itself.

In [ ]:
DATE_COL = "date"

# Used only to group rows into gap-free per-entity series (see Section 4).
# segment_id comes from the preprocessing pipeline and marks where a gap
# longer than a few days was too large to trust as continuous. Neither
# column is a model input.
GROUP_COLS = ["station_id", "segment_id"]

TARGET = "pm25"

# Static covariate: station_id alone uniquely identifies the physical
# location (confirmed with user 2026-09-12 - no lat/long/province).
STATIC_FEATURES = ["station_id"]

# Known-future covariates: deterministically knowable for any date, with no
# dependence on a weather forecast feed that doesn't exist in this dataset.
# wind_direction_*_sin/cos were moved OUT of this group relative to
# feature_manifest.yaml's TFT_KNOWN_TIME_FEATURES view - see Section 2.
FUTURE_FEATURES = [
    "year_index",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos",
    "doy_sin",
    "doy_cos",
]

# Observed (past-only) covariates: meteorology + fire-activity signals only
# known after the fact. sunshine_duration was checked and dropped (see
# Section 2) - 100% missing for 80/537 real PM2.5-bearing groups.
PAST_FEATURES = [
    "temperature_avg",
    "temperature_range",
    "humidity_avg",
    "humidity_range",
    "pressure_avg",
    "pressure_range",
    "wind_speed_avg",
    "wind_direction_avg_sin",
    "wind_direction_avg_cos",
    "log1p_rainfall",
    "rain_event",
    "hotspot_present",
    "log1p_hotspot_count",
    "log1p_hotspot_frp_sum",
]

# Columns that must be numeric floats (everything read from the CSV as
# object/str due to blank cells needs coercing before TimeSeries creation).
NUMERIC_FEATURES = FUTURE_FEATURES + PAST_FEATURES + [TARGET]

REQUIRED_COLUMNS = [DATE_COL] + GROUP_COLS + STATIC_FEATURES + FUTURE_FEATURES + PAST_FEATURES + [TARGET]

FREQ = "D"


## 3. Configuration

Edit these before running. `DATA_PATH` assumes this notebook lives in
`pm25-tft-model/`, next to the CSV's parent directory.

In [ ]:
# --- Paths ---
DATA_PATH = "../clean-data_preprocess_all_stations_daily.csv"
OUTPUT_DIR = Path("artifacts")

# --- Train/val split ---
VAL_HOLDOUT_DAYS = 60  # used only if VAL_START_DATE is None
VAL_START_DATE = None  # e.g. "2026-07-06"; None = last VAL_HOLDOUT_DAYS days of the dataset

# --- Data-quality thresholds (see Section 2) ---
MAX_TARGET_NAN_FRAC = 0.3  # drop a group if pm25 is missing more than this fraction (no-sensor stations)

# --- TFT architecture (see ../REFERENCE.md §2 for what each maps to in the paper) ---
INPUT_CHUNK_LENGTH = 28   # days of history the encoder sees
OUTPUT_CHUNK_LENGTH = 7   # forecast horizon (confirmed: one multi-step model, 1-7 days)
HIDDEN_SIZE = 32
LSTM_LAYERS = 1
NUM_ATTENTION_HEADS = 4
DROPOUT = 0.2
HIDDEN_CONTINUOUS_SIZE = 8
QUANTILES = [0.1, 0.5, 0.9]  # matches the TFT paper's quantile set

# --- Training loop ---
N_EPOCHS = 30
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
GRADIENT_CLIP_VAL = 1.0
ACCELERATOR = "auto"  # PyTorch Lightning accelerator: auto/cpu/gpu/mps
RANDOM_STATE = 42

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 4. Data loading & per-`(station_id, segment_id)` series construction

Pipeline, per `(station_id, segment_id)` group (a group is one gap-free
daily series - `segment_id` comes from the upstream pipeline, incremented
wherever a gap longer than `max_short_gap_days` made a station's history
untrustworthy to treat as continuous):

1. Load the wide CSV, keep only the columns selected in Section 2.
2. Group by `(station_id, segment_id)`; drop groups whose `pm25` is mostly
   missing (stations with no PM2.5 sensor at all), then interpolate any
   remaining short gaps in the rest and drop the rare group where an entire
   covariate column is missing for the whole span.
3. Drop groups too short to form even one
   (`INPUT_CHUNK_LENGTH + OUTPUT_CHUNK_LENGTH`) training window.
4. Attach `station_id` as a static covariate and split each group's target
   series at the validation start date into a train part and an optional
   val part.
5. Fit one `Scaler` per group per series-block (target / past / future) on
   the *train* portion only, then transform the whole group series with it
   - this avoids leaking validation-period statistics into training data.
6. Fit `StaticCovariatesTransformer` once on the training target series
   (label-encodes `station_id` for the TFT's categorical embedding) and
   apply the same fitted transformer to the validation target series.

In [ ]:
@dataclass
class GroupSeries:
    station_id: str
    segment_id: str
    train_target: TimeSeries
    val_target: TimeSeries | None
    past_covariates: TimeSeries  # full span, already scaled
    future_covariates: TimeSeries  # full span, already scaled


@dataclass
class Datasets:
    train_targets: list[TimeSeries]
    train_past_covariates: list[TimeSeries]
    train_future_covariates: list[TimeSeries]
    val_targets: list[TimeSeries]
    val_past_covariates: list[TimeSeries]
    val_future_covariates: list[TimeSeries]
    target_scalers: dict[tuple[str, str], Scaler]
    past_scalers: dict[tuple[str, str], Scaler]
    future_scalers: dict[tuple[str, str], Scaler]
    static_covariates_transformer: StaticCovariatesTransformer
    n_stations: int
    dropped_groups: int
    dropped_no_sensor_groups: int
    dropped_residual_nan_groups: int


def load_raw(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path, usecols=REQUIRED_COLUMNS, parse_dates=[DATE_COL])
    df["station_id"] = df["station_id"].astype(str)
    df["segment_id"] = df["segment_id"].astype(str)
    for col in NUMERIC_FEATURES:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.sort_values(GROUP_COLS + [DATE_COL]).reset_index(drop=True)
    return df


def _has_nan(ts: TimeSeries) -> bool:
    return bool(np.isnan(ts.values(copy=False)).any())


def _fill_gaps(ts: TimeSeries) -> TimeSeries:
    # limit_direction="both" also covers the handful of groups whose very
    # first or last row is NaN - plain linear interpolation only fills gaps
    # strictly between two known values and would leave those edges NaN.
    return MissingValuesFiller(
        interpolate_kwargs={"method": "linear", "limit_direction": "both"}
    ).transform(ts)


In [ ]:
def _build_group_series(
    df: pd.DataFrame,
    input_chunk_length: int,
    output_chunk_length: int,
    val_start_date: pd.Timestamp,
    max_target_nan_frac: float,
) -> tuple[list[GroupSeries], int, int, int]:
    min_len = input_chunk_length + output_chunk_length
    groups: list[GroupSeries] = []
    dropped = 0
    dropped_no_sensor = 0
    dropped_residual_nan = 0

    for (station_id, segment_id), g in df.groupby(GROUP_COLS, sort=False):
        g = g.sort_values(DATE_COL)
        if len(g) < min_len:
            dropped += 1
            continue

        # Some station_ids in this dataset are pure meteorological stations
        # ("... Weather Observing Station" / "... Automatic Weather
        # Station") with no PM2.5 sensor at all - their pm25 column is
        # ~90-100% NaN for the whole segment, not just short measurement
        # gaps. MissingValuesFiller cannot interpolate that (nothing to
        # interpolate from), so these must be excluded as targets here,
        # not silently filled.
        if g[TARGET].isna().mean() > max_target_nan_frac:
            dropped_no_sensor += 1
            continue

        try:
            target_full = TimeSeries.from_dataframe(
                g,
                time_col=DATE_COL,
                value_cols=[TARGET],
                freq=FREQ,
                static_covariates=pd.DataFrame({"station_id": [station_id]}),
            )
            past_full = TimeSeries.from_dataframe(
                g, time_col=DATE_COL, value_cols=PAST_FEATURES, freq=FREQ
            )
            future_full = TimeSeries.from_dataframe(
                g, time_col=DATE_COL, value_cols=FUTURE_FEATURES, freq=FREQ
            )
        except ValueError as exc:
            logger.warning("Skipping station=%s segment=%s: %s", station_id, segment_id, exc)
            dropped += 1
            continue

        target_full = _fill_gaps(target_full)
        past_full = _fill_gaps(past_full)
        future_full = _fill_gaps(future_full)

        # Linear interpolation cannot recover a component that is missing
        # for a group's *entire* span (e.g. an occasional weather sensor
        # outage covering a whole segment). Skip those rather than pass
        # NaN into the model.
        if _has_nan(target_full) or _has_nan(past_full) or _has_nan(future_full):
            logger.warning(
                "Skipping station=%s segment=%s: NaN remained after gap-filling "
                "(likely a fully-missing column for this segment).",
                station_id, segment_id,
            )
            dropped_residual_nan += 1
            continue

        if target_full.end_time() < val_start_date:
            train_target, val_target = target_full, None
        elif target_full.start_time() >= val_start_date:
            train_target, val_target = None, target_full
        else:
            train_target, val_target = target_full.split_before(val_start_date)

        if train_target is None or len(train_target) < min_len:
            dropped += 1
            continue
        if val_target is not None and len(val_target) < output_chunk_length:
            val_target = None

        groups.append(
            GroupSeries(
                station_id=station_id,
                segment_id=segment_id,
                train_target=train_target,
                val_target=val_target,
                past_covariates=past_full,
                future_covariates=future_full,
            )
        )

    return groups, dropped, dropped_no_sensor, dropped_residual_nan


def _scale_group_series(
    groups: list[GroupSeries], val_start_date: pd.Timestamp
) -> tuple[dict, dict, dict]:
    target_scalers, past_scalers, future_scalers = {}, {}, {}

    for grp in groups:
        key = (grp.station_id, grp.segment_id)

        target_scaler = Scaler()
        grp.train_target = target_scaler.fit_transform(grp.train_target)
        if grp.val_target is not None:
            grp.val_target = target_scaler.transform(grp.val_target)
        target_scalers[key] = target_scaler

        # keep_point=False mirrors split_before()'s convention that
        # val_start_date itself belongs to the validation side, so the
        # scaler never sees a validation-period value while fitting.
        past_train_slice = grp.past_covariates.drop_after(val_start_date, keep_point=False)
        if len(past_train_slice) == 0:
            past_train_slice = grp.past_covariates
        past_scaler = Scaler()
        past_scaler.fit(past_train_slice)
        grp.past_covariates = past_scaler.transform(grp.past_covariates)
        past_scalers[key] = past_scaler

        future_train_slice = grp.future_covariates.drop_after(val_start_date, keep_point=False)
        if len(future_train_slice) == 0:
            future_train_slice = grp.future_covariates
        future_scaler = Scaler()
        future_scaler.fit(future_train_slice)
        grp.future_covariates = future_scaler.transform(grp.future_covariates)
        future_scalers[key] = future_scaler

    return target_scalers, past_scalers, future_scalers


def prepare_datasets(
    csv_path: str,
    input_chunk_length: int,
    output_chunk_length: int,
    val_start_date,
    max_target_nan_frac: float = 0.3,
) -> Datasets:
    df = load_raw(csv_path)
    val_start_date = pd.Timestamp(val_start_date)

    groups, dropped, dropped_no_sensor, dropped_residual_nan = _build_group_series(
        df, input_chunk_length, output_chunk_length, val_start_date, max_target_nan_frac
    )
    if not groups:
        raise ValueError(
            "No (station_id, segment_id) group had enough history for "
            f"input_chunk_length={input_chunk_length} + "
            f"output_chunk_length={output_chunk_length}, after excluding "
            "groups without a usable pm25 target."
        )
    logger.info(
        "Built %d group series, dropped %d (too short), %d (no usable pm25 sensor), "
        "%d (unfillable NaN in a fully-missing column).",
        len(groups), dropped, dropped_no_sensor, dropped_residual_nan,
    )

    target_scalers, past_scalers, future_scalers = _scale_group_series(groups, val_start_date)

    train_targets = [grp.train_target for grp in groups]
    static_transformer = StaticCovariatesTransformer()
    train_targets = static_transformer.fit_transform(train_targets)

    val_groups = [grp for grp in groups if grp.val_target is not None]
    val_targets = static_transformer.transform([grp.val_target for grp in val_groups])

    for grp, transformed in zip(groups, train_targets):
        grp.train_target = transformed
    for grp, transformed in zip(val_groups, val_targets):
        grp.val_target = transformed

    # Upper bound on distinct station_id categories the embedding table
    # needs to cover; some of these stations have no usable pm25 sensor and
    # were dropped above, so the encoder may see fewer than this in
    # practice - that's fine, it just leaves a few unused embedding rows.
    n_stations = df["station_id"].nunique()

    return Datasets(
        train_targets=[grp.train_target for grp in groups],
        train_past_covariates=[grp.past_covariates for grp in groups],
        train_future_covariates=[grp.future_covariates for grp in groups],
        val_targets=[grp.val_target for grp in val_groups],
        val_past_covariates=[grp.past_covariates for grp in val_groups],
        val_future_covariates=[grp.future_covariates for grp in val_groups],
        target_scalers=target_scalers,
        past_scalers=past_scalers,
        future_scalers=future_scalers,
        static_covariates_transformer=static_transformer,
        n_stations=n_stations,
        dropped_groups=dropped,
        dropped_no_sensor_groups=dropped_no_sensor,
        dropped_residual_nan_groups=dropped_residual_nan,
    )


## 5. Build the datasets

In [ ]:
if VAL_START_DATE is None:
    _max_date = pd.read_csv(DATA_PATH, usecols=["date"], parse_dates=["date"])["date"].max()
    val_start_date = _max_date - pd.Timedelta(days=VAL_HOLDOUT_DAYS)
    logger.info("No VAL_START_DATE given; using %s (last %d days).", val_start_date.date(), VAL_HOLDOUT_DAYS)
else:
    val_start_date = pd.Timestamp(VAL_START_DATE)

logger.info("Loading data from %s ...", DATA_PATH)
datasets = prepare_datasets(
    csv_path=DATA_PATH,
    input_chunk_length=INPUT_CHUNK_LENGTH,
    output_chunk_length=OUTPUT_CHUNK_LENGTH,
    val_start_date=val_start_date,
    max_target_nan_frac=MAX_TARGET_NAN_FRAC,
)
logger.info(
    "Prepared %d training series (%d with a validation segment), %d stations. "
    "Dropped: %d too short, %d no usable pm25 sensor, %d unfillable NaN.",
    len(datasets.train_targets),
    len(datasets.val_targets),
    datasets.n_stations,
    datasets.dropped_groups,
    datasets.dropped_no_sensor_groups,
    datasets.dropped_residual_nan_groups,
)


## 6. Define and train the `TFTModel`

`categorical_embedding_sizes={"station_id": n_stations}` lets Darts build
the per-station embedding (see `../REFERENCE.md` §3 for the
`StaticCovariatesTransformer` + `categorical_embedding_sizes` pattern this
follows).

In [ ]:
model = TFTModel(
    input_chunk_length=INPUT_CHUNK_LENGTH,
    output_chunk_length=OUTPUT_CHUNK_LENGTH,
    hidden_size=HIDDEN_SIZE,
    lstm_layers=LSTM_LAYERS,
    num_attention_heads=NUM_ATTENTION_HEADS,
    dropout=DROPOUT,
    hidden_continuous_size=HIDDEN_CONTINUOUS_SIZE,
    categorical_embedding_sizes={"station_id": datasets.n_stations},
    likelihood=QuantileRegression(quantiles=QUANTILES),
    batch_size=BATCH_SIZE,
    n_epochs=N_EPOCHS,
    optimizer_kwargs={"lr": LEARNING_RATE},
    random_state=RANDOM_STATE,
    pl_trainer_kwargs={
        "accelerator": ACCELERATOR,
        "gradient_clip_val": GRADIENT_CLIP_VAL,
    },
    add_relative_index=False,
    use_static_covariates=True,
    model_name="pm25_tft",
    save_checkpoints=False,
    force_reset=True,
)

val_kwargs = {}
if datasets.val_targets:
    val_kwargs = {
        "val_series": datasets.val_targets,
        "val_past_covariates": datasets.val_past_covariates,
        "val_future_covariates": datasets.val_future_covariates,
    }
else:
    logger.warning("No validation series available; training without a validation set.")


In [ ]:
logger.info("Fitting TFTModel for %d epochs ...", N_EPOCHS)
model.fit(
    series=datasets.train_targets,
    past_covariates=datasets.train_past_covariates,
    future_covariates=datasets.train_future_covariates,
    **val_kwargs,
)


## 7. Save artifacts

- `pm25_tft_model.pt` (+ Darts' companion files) - the trained model,
  loadable via `TFTModel.load("artifacts/pm25_tft_model.pt")`.
- `scalers.pkl` - a pickle with `target_scalers`, `past_scalers`,
  `future_scalers` (each a `dict[(station_id, segment_id), Scaler]`) and the
  fitted `static_covariates_transformer`, needed to inverse-transform
  predictions back to µg/m³ and to encode new data consistently at
  inference time.

In [ ]:
model_path = OUTPUT_DIR / "pm25_tft_model.pt"
model.save(str(model_path))
logger.info("Saved model to %s", model_path)

scalers_path = OUTPUT_DIR / "scalers.pkl"
with open(scalers_path, "wb") as f:
    pickle.dump(
        {
            "target_scalers": datasets.target_scalers,
            "past_scalers": datasets.past_scalers,
            "future_scalers": datasets.future_scalers,
            "static_covariates_transformer": datasets.static_covariates_transformer,
        },
        f,
    )
logger.info("Saved per-series scalers to %s", scalers_path)


## Known limitations / things to revisit

- Not every `station_id` has a PM2.5 sensor - some are pure meteorological
  stations. Groups whose `pm25` is more than `MAX_TARGET_NAN_FRAC` (default
  30%) missing are dropped before they can become a training target; see
  Section 2, "Target availability." A handful of remaining groups get
  dropped too because one of their weather columns is fully missing for
  that whole segment and can't be interpolated from nothing.
- Series are split into independent `(station_id, segment_id)` groups
  wherever the upstream pipeline detected a gap too long to trust as
  continuous. A station with multiple segments gets one `Scaler` per
  segment rather than one per station, so its segments are normalized
  independently - usually harmless, but worth knowing if a station's
  segments look inconsistently scaled.
- Remaining short gaps in the selected covariates (median residual
  missingness ~0.3% across real PM2.5-bearing groups) are filled with
  `MissingValuesFiller` per group over the *entire* group span before the
  train/val split, so a small amount of interpolation could technically
  draw on a post-cutoff value to fill a pre-cutoff gap. Given how sparse
  these residual gaps are, this is a minor simplification rather than a
  design decision to revisit first.
- No evaluation/backtesting cell is included yet - `model.historical_forecasts()`
  / `model.backtest()` from Darts are the natural next step, using
  `scalers.pkl` to inverse-transform back to µg/m³ before computing error
  metrics.